In [3]:
from pathlib import Path
import pandas as pd


In [4]:
YEAR = 2024

RAW_DIR = Path(f"data/raw/{YEAR}")
PROCESSED_DIR = Path("data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


all_races = []


# ============================================================
# RECORRER LOS 24 GP
# ============================================================

race_dirs = sorted(
    [p for p in RAW_DIR.iterdir() if p.is_dir()]
)


for race_dir in race_dirs:

    race_name = race_dir.name

    race_file = race_dir / "R.parquet"

    if not race_file.exists():
        print(f"[WARNING] No existe: {race_file}")
        continue

    print(f"Procesando {race_name}...")

    df = pd.read_parquet(race_file)

    # --------------------------------------------------------
    # Información del GP
    # --------------------------------------------------------

    df["Race"] = race_name

    # Extraemos el número de ronda

    # --------------------------------------------------------
    # Columnas que queremos conservar
    # --------------------------------------------------------

    columns = [
        "Race",
        "Driver",
        "DriverNumber",
        "Team",
        "LapNumber",
        "LapTime",
        "Compound",
        "TyreLife",
        "FreshTyre",
        "Stint",
        "Position",
        "PitInTime",
        "PitOutTime",
        "TrackStatus",
        "IsAccurate",
        "Deleted",
    ]

    # Nos quedamos solo con las columnas disponibles
    columns = [
        col for col in columns
        if col in df.columns
    ]

    df = df[columns].copy()

    # --------------------------------------------------------
    # Convertir LapTime a segundos
    # --------------------------------------------------------

    df["LapTimeSeconds"] = (
        df["LapTime"]
        .dt.total_seconds()
    )

    # --------------------------------------------------------
    # Limpieza básica
    # --------------------------------------------------------

    # Vueltas eliminadas
    df = df[df["Deleted"] == False]

    # Vueltas que FastF1 considera precisas
    df = df[df["IsAccurate"] == True]

    # Necesitamos tiempo de vuelta
    df = df[df["LapTimeSeconds"].notna()]

    # Necesitamos número de vuelta
    df = df[df["LapNumber"].notna()]

    # --------------------------------------------------------
    # Orden
    # --------------------------------------------------------

    df = df.sort_values(
        ["Driver", "LapNumber"]
    )

    all_races.append(df)


# ============================================================
# UNIR TODAS LAS CARRERAS
# ============================================================

race_laps = pd.concat(
    all_races,
    ignore_index=True
)


# ============================================================
# GUARDAR
# ============================================================

output_file = (
    PROCESSED_DIR /
    "race_laps_2024.parquet"
)

race_laps.to_parquet(
    output_file,
    index=False
)


# ============================================================
# INFORMACIÓN FINAL
# ============================================================

print("\n" + "=" * 70)
print("DATASET DE CARRERAS CREADO")
print("=" * 70)

print(f"Carreras: {race_laps['Race'].nunique()}")
print(f"Vueltas:  {len(race_laps):,}")
print(f"Pilotos:  {race_laps['Driver'].nunique()}")
print(f"Equipos:  {race_laps['Team'].nunique()}")

print("\nColumnas:")
print(race_laps.columns.tolist())

print("\nGuardado en:")
print(output_file)

Procesando Austin...
Procesando Baku...
Procesando Barcelona...
Procesando Budapest...
Procesando Imola...
Procesando Jeddah...
Procesando Las Vegas...
Procesando Lusail...
Procesando Marina Bay...
Procesando Melbourne...
Procesando Mexico City...
Procesando Miami...
Procesando Monaco...
Procesando Montréal...
Procesando Monza...
Procesando Sakhir...
Procesando Shanghai...
Procesando Silverstone...
Procesando Spa-Francorchamps...
Procesando Spielberg...
Procesando Suzuka...
Procesando São Paulo...
Procesando Yas Island...
Procesando Zandvoort...

DATASET DE CARRERAS CREADO
Carreras: 24
Vueltas:  23,256
Pilotos:  24
Equipos:  10

Columnas:
['Race', 'Driver', 'DriverNumber', 'Team', 'LapNumber', 'LapTime', 'Compound', 'TyreLife', 'FreshTyre', 'Stint', 'Position', 'PitInTime', 'PitOutTime', 'TrackStatus', 'IsAccurate', 'Deleted', 'LapTimeSeconds']

Guardado en:
data\processed\race_laps_2024.parquet


In [6]:
race_laps.drop(columns=["LapTime"], inplace=True)
race_laps.head()

,Race,Driver,DriverNumber,Team,LapNumber,Compound,TyreLife,FreshTyre,Stint,Position,PitInTime,PitOutTime,TrackStatus,IsAccurate,Deleted,LapTimeSeconds
0,Austin,ALB,23,Williams,2.0,MEDIUM,2.0,True,1.0,18.0,NaT,NaT,12,True,False,104.544
1,Austin,ALB,23,Williams,6.0,MEDIUM,3.0,True,2.0,19.0,NaT,NaT,1,True,False,104.652
2,Austin,ALB,23,Williams,7.0,MEDIUM,4.0,True,2.0,19.0,NaT,NaT,1,True,False,102.627
3,Austin,ALB,23,Williams,8.0,MEDIUM,5.0,True,2.0,19.0,NaT,NaT,1,True,False,102.144
4,Austin,ALB,23,Williams,9.0,MEDIUM,6.0,True,2.0,19.0,NaT,NaT,1,True,False,102.001


In [7]:
df = race_laps.copy()

In [11]:
print("=" * 70)
print("RACE DATASET 2024")
print("=" * 70)

print(f"\nFilas:       {len(df):,}")
print(f"Carreras:    {df['Race'].nunique()}")
print(f"Circuitos:   {df['Race'].nunique()}")
print(f"Pilotos:     {df['Driver'].nunique()}")
print(f"Equipos:     {df['Team'].nunique()}")


print("\n" + "-" * 70)
print("CIRCUITOS")
print("-" * 70)

print(
    df["Race"]
    .value_counts()
    .sort_index()
)


print("\n" + "-" * 70)
print("COMPUESTOS")
print("-" * 70)

print(
    df["Compound"]
    .value_counts(dropna=False)
)


print("\n" + "-" * 70)
print("VUeltas POR CARRERA")
print("-" * 70)

print(
    df.groupby(
        ["Race"]
    )
    .size()
    .to_string()
)


print("\n" + "-" * 70)
print("VALORES NULOS")
print("-" * 70)

important_columns = [
    "Driver",
    "Team",
    "Race",
    "LapNumber",
    "LapTimeSeconds",
    "Compound",
    "TyreLife",
    "Stint",
    "Position",
]

print(
    df[important_columns]
    .isna()
    .sum()
)


print("\n" + "=" * 70)

RACE DATASET 2024

Filas:       23,256
Carreras:    24
Circuitos:   24
Pilotos:     24
Equipos:     10

----------------------------------------------------------------------
CIRCUITOS
----------------------------------------------------------------------
Race
Austin                902
Baku                  879
Barcelona            1192
Budapest             1249
Imola                1153
Jeddah                800
Las Vegas             820
Lusail                701
Marina Bay           1097
Melbourne             856
Mexico City          1051
Miami                 915
Monaco               1183
Montréal              995
Monza                 914
Sakhir               1005
Shanghai              757
Silverstone           841
Spa-Francorchamps     741
Spielberg            1259
Suzuka                777
São Paulo             924
Yas Island            895
Zandvoort            1350
Name: count, dtype: int64

----------------------------------------------------------------------
COMPUESTOS
------